# Self-test for the 1.2 Gate

Making the whole transformer from scratch

The transformer Architecture:

Inputs → Embedding[Inputs] + Positional Encoding → N blocks [ Norm → Multihead attention → Dropout → Residual → Norm → FFN → Dropout → Residual] → Norm → Output projection

In [1]:
# -- Imports

import torch
import math

In [2]:
class Linear:
    """
    Returns a linear layer:
    y = x @ W + b

    by default, it adds bias: b
    """
    def __init__(self, input_dim, output_dim, seed, bias=True, device='xpu', scale=1.0):
        generator=torch.Generator(device=device).manual_seed(seed)

        self.weight = (
            torch.randn(size=(input_dim, output_dim), device=device, generator=generator)
            * 1/math.sqrt(input_dim)
            * scale
        ).requires_grad_(True)
        if bias:
            self.bias = torch.zeros(output_dim, device=device)

    def __call__(self, x):
        output = x @ self.weight
        if self.bias is not None:
             output += self.bias

        return output

    def parameters(self):
        return [self.weight] + ([self.bias] if self.bias is not None else [])

In [11]:
class Embedding:
    """
    Generates a learnable embedding matrix (E) of shape (vocab_dim, embedding_dim)
    Embeds given inputs(X) by indexing in E as following:
        embedded inputs = E[X]
        shape change: (batch_size, context_size) -> (batch_size, context_size, embedding_dim)
    """
    def __init__(self, vocab_dim, embedding_dim, seed, device='xpu', scale=0.1):
        embedding = Linear(input_dim=vocab_dim, output_dim=embedding_dim, seed=seed, bias=False, device=device, scale=scale)
        self.embedding_matrix = embedding.weight

    def __call__(self, x):
        return self.embedding_matrix[x]

    def parameters(self):
        return [self.embedding_matrix]

In [4]:
class PositionalEncoding:
    """
    Generates a learnable positional encoding matrix of shape (vocab_dim, embedding_dim)
    Adds the positional information(P) to the embedded inputs(E[X]) as follows:
        encoded inputs = E[X] + P
        shape change: None, because it's just simple addition broadcasted over the batch_size
             (batch_size, context_size, embedding_dim) + (context_size, embedding_dim) =(batch_size, context_size, embedding_dim) + (1, context_size, embedding_dim)
    """
    def __init__(self, context_size, embedding_dim, seed, device='xpu', scale=0.1):
        self.positional_encoding_matrix = Linear(input_dim=context_size, output_dim=embedding_dim, seed=seed, bias=False, device=device, scale=scale)

    def __call__(self):
        return self.positional_encoding_matrix.weight

    def parameters(self):
        return [self.positional_encoding_matrix]

In [10]:
class GeLU:
    """
    Simple GeLU activation class.
    Returns GeLU(x) using the tanh approximation:
        GeLU(x) = 0.5x(1+tanh(√2/π(x+0.044715+x³)))
    """
    def __call__(self, x):
        output = 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))

        return output

    @staticmethod
    def parameters(self):
        return []

In [5]:
class LayerNorm:
    """
    Layer Normalization class.
    Normalizes the inputs before scaling with a learnable affine(γ) and adding a learnable bias (β):
        - Normalizes across the feature(embedding) dimension:
            Z = (X-μ(X)) / (σ²(X) + ε)
            ε: to avoid division by zero error
        - Scales with a learnable affine γ and adds a learnable bias β:
            Z = γ * Z + β
    """
    def __init__(self, embedding_dim, device='xpu', epsilon=1e-6):
        self.gamma = torch.ones(embedding_dim, device=device).requires_grad_(True)
        self.beta  = torch.zeros(embedding_dim, device=device).requires_grad_(True)
        self.eps   = epsilon

    def __call__(self, x):
        mean       = x.mean(-1, keepdim=True)
        variance   = ((x-mean)**2).mean(-1, keepdim=True)
        normalized = (x-mean)/torch.sqrt(variance+self.eps)
        return self.gamma * normalized + self.beta

    def parameters(self):
        return [self.gamma, self.beta]

In [6]:
class Dropout:
    """
    Dropout class
    Drops the values in a given parameter with a probability:= p and scales the remaining values with 1/(1-p)
    """
    def __init__(self, p, flag='Train'):
        self.p = p
        self.flag = flag

    def __call__(self, x):
        if self.p == 1:
            return torch.zeros_like(x)

        if self.flag == 'Train':
            self.mask = torch.rand_like(x) > self.p
            return (x * self.mask) / (1.0-self.p)

        return x

    @staticmethod
    def parameters(self):
        return []

In [7]:
class MultiheadAttention:
    """
    Fused multihead attention class.
    For inputs shape: (batch_size, context_size, embedding_dimension)
    Calculates the multihead attention(mha) for inputs(x) as follows:
        creates 4 weight matrices:
            W_Q: shape = (embedding_dimension, dim_qk * num_heads)
            W_K: shape = (embedding_dimension, dim_qk * num_heads)
            W_V: shape = (embedding_dimension, dim_v * num_heads)
            W_O: shape = (dim_v * num_heads, embedding_dimension)

        and calculates the Q/K/V (for all heads) for x:
            Q = x @ W_Q {shape: (batch_size, context_size, dim_qk * num_heads)}
            K = x @ W_K {shape: (batch_size, context_size, dim_qk * num_heads)}
            V = x @ W_V {shape: (batch_size, context_size, dim_v * num_heads)}

            before reshaping the Q/K/V into individual Q/K/V matrices:
                Q: new_shape: (batch_size, num_heads, context_size, dim_qk)
                K: new_shape: (batch_size, num_heads, context_size, dim_qk)
                V: new_shape: (batch_size, num_heads, context_size, dim_v)

        then calculates the scaled scores for all heads together:
            scaled_scores = Q @ K.T / √dim_qk {shape: (batch_size, num_heads, context_size, context_size)}

        applies softmax and dropout on the scaled scores:
            final_scores = Dropout ( softmax (scaled_scores) ) {shape: (batch_size, num_heads, context_size, context_size)}

        projects the final_scores into values space:
            projection = (final_scores @ V) {shape: (batch_size, num_heads, context_size, dim_v)}

        converts the projection back into a single multihead:
            projection: new_shape = (batch_size, context_size, dim_v * num_heads)

        returns a final projection back into embedding space:
            final projection = projection @ W_O {shape: (batch_size, context_size, embedding_dimension)}
    """
    def __init__(self, context_size, flag, embedding_dim, dim_qk, dim_v, num_heads=4, device='xpu', seed=42, scale=1):
        self.query_weight = Linear(input_dim=embedding_dim, output_dim=dim_qk*num_heads, seed=seed, bias=False, device=device, scale=scale)
        self.key_weight   = Linear(input_dim=embedding_dim, output_dim=dim_qk*num_heads, seed=seed, bias=False, device=device, scale=scale)
        self.value_weight = Linear(input_dim=embedding_dim, output_dim=dim_v*num_heads, seed=seed, bias=False, device=device, scale=scale)
        self.out_weight   = Linear(input_dim=dim_v*num_heads, output_dim=embedding_dim, seed=seed, bias=False, device=device, scale=scale)

        self.causal_mask = torch.triu(
            torch.ones((1, 1, context_size, context_size), device=device),
            diagonal=1
        ).bool()

        self.attn_dropout = Dropout(p=0.2, flag=flag) if flag is not None else None

        self.context_size = context_size
        self.num_heads    = num_heads
        self.dim_k        = dim_qk
        self.dim_v        = dim_v

    def __call__(self, x):
        batch_size = x.shape[0]
        Query = ((x @ self.query_weight.weight).reshape(batch_size, self.context_size, self.num_heads, self.dim_k)).transpose(1, 2)
        Key   = ((x @ self.key_weight.weight).reshape(batch_size, self.context_size, self.num_heads, self.dim_k)).transpose(1, 2)
        Value = (x @ self.value_weight.weight).reshape(batch_size, self.context_size, self.num_heads, self.dim_v).transpose(1, 2)

        scaled_scores = (Query @ Key.transpose(-2, -1)) / math.sqrt(self.dim_k)
        masked_scores = scaled_scores.masked_fill(self.causal_mask, float('-inf'))

        A = torch.exp(masked_scores - torch.max(masked_scores, dim=-1, keepdim=True)[0]) / torch.sum(torch.exp(masked_scores - torch.max(masked_scores, dim=-1, keepdim=True)[0]), dim=-1, keepdim=True)

        if self.attn_dropout is not None:
            A = self.attn_dropout(A)

        out = (A @ Value).transpose(1, 2).reshape(batch_size, self.context_size, self.num_heads*self.dim_v)

        return self.out_weight(out)

    def parameters(self):
        return [weight.parameters() for weight in [self.query_weight, self.key_weight, self.value_weight, self.out_weight]]

In [8]:
class FeedForward:
    """
    Simple feed forward network with non-linear activation: GeLU [using tanh approximation]

    Takes the inputs (x₀) and applies the following transformations:
        1. x₁ = x₀ @ W₁ + b₁
        2. x' = GeLU(x₁)
        3. x  = x' @ W₂ + b₂

    Returns x
    """
    def __init__(self, scale, input_dim, ffn_dim, seed, device='xpu'):
        self.ffn_in  = Linear(input_dim, ffn_dim, seed, bias=True, device=device, scale=scale)
        self.gelu    = GeLU()
        self.ffn_out = Linear(ffn_dim, input_dim, seed, bias=True, device=device, scale=scale)

    def __call__(self, x):
        return self.ffn_out(self.gelu(self.ffn_in(x)))

    def parameters(self):
        return [self.ffn_in.parameters() + self.gelu.parameters() + self.ffn_out.parameters()]

In [ ]:
class Block:
    """
    1 Block of the transformer: [Layer Norm -> Multihead Attention -> Dropout -> Residual -> Layer Norm -> Feed Forward Network -> Dropout -> Residual]
    Takes the input (x) and applies the following transformations:
        Multihead attention half:
            1. x₁ = Layernorm(x)
            2. x₂ = Multihead attention(x₁)
            3. x₃ = Dropout(x₂)
            4. x' = x + x₃
        Feed forward half:
            5. x₅ = Layernorm(x')
            6. x₆ = Feed forward(x₅)
            7. x₇ = Dropout(x₆)
            8. x₀ = x + x₇
    Returns x + x₀ of the same shape as x
    """
    def __init__(self, flag, block_initialization_scale,
            embedding_dim, context_size,
            dim_qk, dim_v, num_heads,
            ffn_dim,
            p_attn, p_ffn,
            seed, device
    ):
        self.attn_layernorm    = LayerNorm(embedding_dim=embedding_dim, device=device)
        self.multihead_attn    = MultiheadAttention(context_size=context_size, flag=flag, embedding_dim=embedding_dim, dim_qk=dim_qk, dim_v=dim_v, num_heads=num_heads, device=device, seed=seed, scale=block_initialization_scale)

        self.ffn_layernorm     = LayerNorm(embedding_dim=embedding_dim, device=device)
        self.feedforward       = FeedForward(scale=block_initialization_scale,input_dim=embedding_dim, ffn_dim=ffn_dim, seed=seed, device=device)

        self.attn_dropout      = Dropout(p=p_attn, flag=flag)
        self.ffn_dropout      = Dropout(p=p_ffn, flag=flag)

    def __call__(self, x):
        attn_out = self.attn_dropout(self.multihead_attn(self.attn_layernorm(x)))
        ffn_out  = self.ffn_dropout(self.feedforward(self.ffn_layernorm(x + attn_out)))

        return x + ffn_out + attn_out

    def parameters(self):
        return [parameter.parameters() for parameter in [self.attn_layernorm, self.ffn_layernorm, self.feedforward, self.multihead_attn, self.attn_dropout, self.ffn_dropout]]

In [ ]:
class Transformer:
    """
    Full Transformer
    """
    def __init__(self, flag,
            vocab_dim, embedding_dim, context_size,
            embedding_scale, positional_encoding_scale,
            dim_qk, dim_v, num_heads,
            ffn_dim,
            p_embedding, p_attn, p_ffn, num_blocks,
            layernorm_epsilon,
            seed, device
    ):
        block_initialization_scale = 1/math.sqrt(2*num_blocks)
        self.embedding = Embedding(vocab_dim=vocab_dim, embedding_dim=embedding_dim, seed=seed, device=device, scale=embedding_scale)
        self.positional_encoding = PositionalEncoding(context_size=context_size, embedding_dim=embedding_dim, seed=seed, device=device, scale=positional_encoding_scale)

        self.blocks = [
            Block(
                flag=flag, block_initialization_scale=block_initialization_scale,
            embedding_dim=embedding_dim, context_size=context_size,
            dim_qk=dim_qk, dim_v=dim_v, num_heads=num_heads,
            ffn_dim=ffn_dim,
            p_attn=p_attn, p_ffn=p_ffn,
            seed=seed, device=device
            ) for _ in range(num_blocks)
        ]

        self.embedding_dropout = Dropout(p=p_embedding, flag=flag)
        self.final_layernorm = LayerNorm(embedding_dim=embedding_dim, device=device, epsilon=layernorm_epsilon)

    def __call__(self, x):
        embedded_inputs         = self.embedding(x)
        encoded_inputs          = embedded_inputs + self.positional_encoding()
        representation  = self.embedding_dropout(encoded_inputs)

        for block in self.blocks:
            representation = block(representation)

        final_representation = self.final_layernorm(representation)
        logits = final_representation @ self.embedding.embedding_matrix.T

        return logits

    def parameters(self):
        block_parameters = []
        for block in self.blocks:
            block_parameters += block.parameters()

        return [
            self.embedding.parameters(),
            self.positional_encoding.parameters(),
            self.embedding_dropout.parameters(),
            self.final_layernorm.parameters(),
            block_parameters
        ]